# Fabric SQL Pools Configuration — Inventory & Enablement

Loops over every Fabric workspace you can see (with optional **allow** / **exclude** lists) and:

1. Calls [`GET .../warehouses/sqlPoolsConfiguration?beta=True`](https://learn.microsoft.com/en-us/rest/api/fabric/warehouse/sql-pools/get-sql-pools-configuration(beta)) to read whether
   custom SQL pools are enabled and, if so, every pool's `optimizeForReads`, `maxResourcePercentage`,
   `isDefault` and `classifier` values — saved to a **Delta table** so you can query it later.
2. Optionally calls [`PATCH .../warehouses/sqlPoolsConfiguration?beta=True`](https://learn.microsoft.com/en-us/rest/api/fabric/warehouse/sql-pools/update-sql-pools-configuration(beta))
   to **enable** a single default pool (`optimizeForReads = false`, `maxResourcePercentage = 15`) on
   workspaces where custom SQL pools are not already enabled.

Every HTTP call — success or failure — is appended to an API log Delta table.

### Before you run
* You must hold the **Admin** workspace role on each workspace (the API returns `401/403` otherwise — that is logged, not fatal).
* Attach a **default Lakehouse** to this notebook (or set `TABLE_BASE_PATH` in the config cell).
* Cell 5 (the enable cell) ships with `DRY_RUN = True`. Review the plan, then set it to `False`.

### Output tables
| Table | Contents |
|---|---|
| `sqlpools_configuration` | One row per workspace/pool: workspace name, enabled flag, pool details |
| `sqlpools_api_log` | One row per REST call: URL, request body, status code, error, duration |
| `sqlpools_enable_actions` | One row per workspace from the enable cell: decision + outcome |


## 1. Configuration
Everything below is a knob. Nothing else in the notebook needs editing.

In [ ]:
# ---------------------------------------------------------------------------
# Fabric REST API
# ---------------------------------------------------------------------------
API_BASE = "https://api.fabric.microsoft.com/v1"
# The SQL pools APIs are in beta and REQUIRE beta=True.
BETA = True

# ---------------------------------------------------------------------------
# Workspace scoping
# ---------------------------------------------------------------------------
# ALLOW_LIST: when non-empty ONLY these workspaces are processed.
#             Accepts workspace display names or workspace IDs (case-insensitive).
#             e.g. ["my-test-workspace", "00000000-1111-2222-3333-444444444444"]
#             Leave EMPTY to process EVERY workspace you can see - start with a
#             couple of test workspaces instead.
ALLOW_LIST = ["my-test-workspace"]

# EXCLUDE_LIST: these workspaces are always skipped. Applied AFTER the allow list.
#               Accepts workspace display names or workspace IDs (case-insensitive).
EXCLUDE_LIST = []

# Workspace types to consider. "Personal" and "AdminWorkspace" have no warehouses.
INCLUDE_WORKSPACE_TYPES = ["Workspace"]

# Optional: only workspaces on these capacity IDs. Empty = any capacity.
INCLUDE_CAPACITY_IDS = []

# ---------------------------------------------------------------------------
# Output Delta tables
# ---------------------------------------------------------------------------
INVENTORY_TABLE = "sqlpools_configuration"
LOG_TABLE = "sqlpools_api_log"
ACTION_TABLE = "sqlpools_enable_actions"

# None  -> write into this notebook's default (attached) Lakehouse as managed tables.
# Or set an explicit OneLake folder to write external Delta tables, e.g.
#   "abfss://battest@onelake.dfs.fabric.microsoft.com/test.Lakehouse/Tables"
TABLE_BASE_PATH = None

# "append" keeps the full history of every run. "overwrite" keeps only the latest run.
WRITE_MODE = "append"

# ---------------------------------------------------------------------------
# Pool settings applied by the "enable custom SQL pools" cell
# ---------------------------------------------------------------------------
CUSTOM_SQL_POOLS_ENABLED = True
POOL_NAME = "defaultpool"
POOL_IS_DEFAULT = True
POOL_MAX_RESOURCE_PERCENTAGE = 15
POOL_OPTIMIZE_FOR_READS = False
# Optional classifier, e.g. {"type": "Application Name", "value": ["ETL", "Load"]}
POOL_CLASSIFIER = None

# SAFETY: True = print the plan, call nothing. Set to False to actually PATCH.
DRY_RUN = True
# True = re-apply the settings even on workspaces where pools are already enabled.
# False = if custom SQL pools are already enabled, do nothing (default).
FORCE_UPDATE = False

# ---------------------------------------------------------------------------
# HTTP behaviour
# ---------------------------------------------------------------------------
MAX_RETRIES = 5
RETRY_BACKOFF_SECONDS = 5
REQUEST_TIMEOUT = 60
# Truncate stored response bodies so one huge error cannot bloat the log table.
MAX_LOGGED_BODY_CHARS = 8000

## 2. Helpers — auth, REST client with retry/logging, Delta writer

In [ ]:
import datetime
import json
import time
import uuid
from urllib.parse import quote

import requests
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)
print(f"run_id = {RUN_ID}   started (utc) = {RUN_TS:%Y-%m-%d %H:%M:%S}")


def utcnow():
    return datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)

# --- token -----------------------------------------------------------------
_TOKEN = {"value": None, "acquired_at": 0.0}


def _acquire_token():
    """Fabric REST API token for the identity running the notebook."""
    try:
        import notebookutils

        return notebookutils.credentials.getToken("pbi")
    except Exception:
        from notebookutils import mssparkutils

        return mssparkutils.credentials.getToken("pbi")


def auth_headers(force_refresh=False):
    if force_refresh or _TOKEN["value"] is None or (time.time() - _TOKEN["acquired_at"]) > 1800:
        _TOKEN["value"] = _acquire_token()
        _TOKEN["acquired_at"] = time.time()
    return {"Authorization": f"Bearer {_TOKEN['value']}", "Content-Type": "application/json"}


# --- REST client -----------------------------------------------------------
API_LOG = []  # every call made in this session, flushed to LOG_TABLE


def _extract_error(payload):
    """Pull errorCode / message out of a Fabric ErrorResponse body."""
    if not isinstance(payload, dict):
        return None, None
    nested = payload.get("error") if isinstance(payload.get("error"), dict) else {}
    code = payload.get("errorCode") or nested.get("code")
    message = payload.get("message") or nested.get("message")
    return code, message


def fabric_request(method, url, body=None, operation=None, workspace_id=None, workspace_name=None):
    """Call the Fabric REST API. Retries 429/5xx (honouring Retry-After) and logs the result."""
    started = time.time()
    attempt = 0
    status = payload = raw_text = err_code = err_msg = None

    while attempt < MAX_RETRIES:
        attempt += 1
        try:
            response = requests.request(
                method, url, headers=auth_headers(), json=body, timeout=REQUEST_TIMEOUT
            )
            status = response.status_code
            raw_text = response.text or ""
            try:
                payload = response.json() if raw_text.strip() else {}
            except ValueError:
                payload = None

            if status == 401 and attempt < MAX_RETRIES:
                auth_headers(force_refresh=True)
                continue
            if (status == 429 or 500 <= status < 600) and attempt < MAX_RETRIES:
                wait = int(response.headers.get("Retry-After") or RETRY_BACKOFF_SECONDS * attempt)
                time.sleep(wait)
                continue
            break
        except Exception as ex:  # network / timeout
            status, payload, raw_text = None, None, None
            err_code, err_msg = type(ex).__name__, str(ex)
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)
                continue
            break

    success = status is not None and 200 <= status < 300
    if not success and err_code is None:
        err_code, err_msg = _extract_error(payload)
        if err_msg is None:
            err_msg = (raw_text or "")[:1000] or f"HTTP {status}"

    record = {
        "log_id": str(uuid.uuid4()),
        "run_id": RUN_ID,
        "run_timestamp": RUN_TS,
        "logged_at": utcnow(),
        "workspace_id": workspace_id,
        "workspace_name": workspace_name,
        "operation": operation,
        "method": method.upper(),
        "url": url,
        "request_body": json.dumps(body) if body is not None else None,
        "status_code": status,
        "success": success,
        "attempts": attempt,
        "duration_ms": int((time.time() - started) * 1000),
        "error_code": err_code,
        "error_message": (err_msg or None) if not success else None,
        "response_body": (raw_text or "")[:MAX_LOGGED_BODY_CHARS] or None,
        "payload": payload,  # in-memory only, stripped before writing
    }
    API_LOG.append(record)
    return record


# --- Delta helpers ---------------------------------------------------------
def _target(table_name):
    if TABLE_BASE_PATH:
        return None, f"{TABLE_BASE_PATH.rstrip('/')}/{table_name}"
    return table_name, None


def write_delta(df, table_name, mode=None):
    mode = mode or WRITE_MODE
    table, path = _target(table_name)
    writer = df.write.format("delta").mode(mode).option("mergeSchema", "true")
    if mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    if path:
        writer.save(path)
    else:
        writer.saveAsTable(table)
    return path or table


def read_delta(table_name):
    table, path = _target(table_name)
    return spark.read.format("delta").load(path) if path else spark.read.table(table)


def register_views():
    """Expose the output tables as temp views (v_<table>) so the SQL below works
    for both managed tables and explicit abfss paths."""
    created = []
    for name in (INVENTORY_TABLE, LOG_TABLE, ACTION_TABLE):
        try:
            read_delta(name).createOrReplaceTempView(f"v_{name}")
            created.append(f"v_{name}")
        except Exception:
            pass  # table not written yet
    return created


LOG_SCHEMA = StructType(
    [
        StructField("log_id", StringType()),
        StructField("run_id", StringType()),
        StructField("run_timestamp", TimestampType()),
        StructField("logged_at", TimestampType()),
        StructField("workspace_id", StringType()),
        StructField("workspace_name", StringType()),
        StructField("operation", StringType()),
        StructField("method", StringType()),
        StructField("url", StringType()),
        StructField("request_body", StringType()),
        StructField("status_code", IntegerType()),
        StructField("success", BooleanType()),
        StructField("attempts", IntegerType()),
        StructField("duration_ms", IntegerType()),
        StructField("error_code", StringType()),
        StructField("error_message", StringType()),
        StructField("response_body", StringType()),
    ]
)
_LOG_FIELDS = [f.name for f in LOG_SCHEMA.fields]


def flush_api_log():
    """Append everything captured in API_LOG to LOG_TABLE, then clear the buffer."""
    if not API_LOG:
        print("No API calls to log.")
        return None
    rows = [{k: rec.get(k) for k in _LOG_FIELDS} for rec in API_LOG]
    target = write_delta(spark.createDataFrame(rows, schema=LOG_SCHEMA), LOG_TABLE)
    failures = sum(1 for r in rows if not r["success"])
    print(f"Logged {len(rows)} API call(s) ({failures} failed) to {target}")
    API_LOG.clear()
    return target


if not TABLE_BASE_PATH:
    _default_lakehouse = None
    try:
        import notebookutils

        _default_lakehouse = notebookutils.runtime.context.get("defaultLakehouseId")
    except Exception:
        pass
    if not _default_lakehouse:
        raise RuntimeError(
            "No default Lakehouse is attached to this notebook. Attach one from the Explorer "
            "pane, or set TABLE_BASE_PATH to an abfss Tables folder in the config cell."
        )
    print(f"Default lakehouse: {_default_lakehouse}")

print("Helpers ready.")

## 3. Discover workspaces (allow list / exclude list)

In [ ]:
def list_workspaces():
    """Page through GET /v1/workspaces."""
    workspaces, url = [], f"{API_BASE}/workspaces"
    while url:
        record = fabric_request("GET", url, operation="ListWorkspaces")
        if not record["success"]:
            raise RuntimeError(
                f"Could not list workspaces: HTTP {record['status_code']} {record['error_message']}"
            )
        body = record["payload"] or {}
        workspaces.extend(body.get("value", []))
        token = body.get("continuationToken")
        url = f"{API_BASE}/workspaces?continuationToken={quote(token)}" if token else None
    return workspaces


def _norm(value):
    return (value or "").strip().lower()


_ALLOW = {_norm(v) for v in ALLOW_LIST if _norm(v)}
_EXCLUDE = {_norm(v) for v in EXCLUDE_LIST if _norm(v)}
_CAPACITIES = {_norm(v) for v in INCLUDE_CAPACITY_IDS if _norm(v)}


def scope_reason(ws):
    """Return None when the workspace is in scope, otherwise why it was skipped."""
    keys = {_norm(ws.get("id")), _norm(ws.get("displayName"))}
    if INCLUDE_WORKSPACE_TYPES and ws.get("type") not in INCLUDE_WORKSPACE_TYPES:
        return f"type '{ws.get('type')}' not in INCLUDE_WORKSPACE_TYPES"
    if _ALLOW and not (keys & _ALLOW):
        return "not in ALLOW_LIST"
    if keys & _EXCLUDE:
        return "in EXCLUDE_LIST"
    if _CAPACITIES and _norm(ws.get("capacityId")) not in _CAPACITIES:
        return "capacity not in INCLUDE_CAPACITY_IDS"
    return None


all_workspaces = list_workspaces()
target_workspaces, skipped_workspaces = [], []
for ws in all_workspaces:
    reason = scope_reason(ws)
    (skipped_workspaces if reason else target_workspaces).append((ws, reason))

print(f"Workspaces visible : {len(all_workspaces)}")
print(f"In scope           : {len(target_workspaces)}")
print(f"Skipped            : {len(skipped_workspaces)}")
for ws, _ in sorted(target_workspaces, key=lambda x: _norm(x[0].get("displayName"))):
    print(f"  - {ws.get('displayName')}  ({ws.get('id')})")

## 4. Read the SQL pools configuration for every in-scope workspace
Writes one row **per pool** (or a single row with null pool columns when a workspace has none)
to the `sqlpools_configuration` Delta table.

In [ ]:
INVENTORY_SCHEMA = StructType(
    [
        StructField("run_id", StringType()),
        StructField("run_timestamp", TimestampType()),
        StructField("workspace_id", StringType()),
        StructField("workspace_name", StringType()),
        StructField("workspace_type", StringType()),
        StructField("capacity_id", StringType()),
        StructField("status_code", IntegerType()),
        StructField("succeeded", BooleanType()),
        StructField("custom_sql_pools_enabled", BooleanType()),
        StructField("pool_count", IntegerType()),
        StructField("pool_name", StringType()),
        StructField("pool_is_default", BooleanType()),
        StructField("pool_max_resource_percentage", IntegerType()),
        StructField("pool_optimize_for_reads", BooleanType()),
        StructField("classifier_type", StringType()),
        StructField("classifier_values", ArrayType(StringType())),
        StructField("error_code", StringType()),
        StructField("error_message", StringType()),
        StructField("raw_response", StringType()),
    ]
)


def sql_pools_url(workspace_id):
    return f"{API_BASE}/workspaces/{workspace_id}/warehouses/sqlPoolsConfiguration?beta={str(BETA).lower()}"


def get_sql_pools_configuration(ws):
    return fabric_request(
        "GET",
        sql_pools_url(ws["id"]),
        operation="GetSqlPoolsConfiguration",
        workspace_id=ws.get("id"),
        workspace_name=ws.get("displayName"),
    )


def inventory_rows(ws, record):
    base = {
        "run_id": RUN_ID,
        "run_timestamp": RUN_TS,
        "workspace_id": ws.get("id"),
        "workspace_name": ws.get("displayName"),
        "workspace_type": ws.get("type"),
        "capacity_id": ws.get("capacityId"),
        "status_code": record["status_code"],
        "succeeded": record["success"],
        "custom_sql_pools_enabled": None,
        "pool_count": None,
        "pool_name": None,
        "pool_is_default": None,
        "pool_max_resource_percentage": None,
        "pool_optimize_for_reads": None,
        "classifier_type": None,
        "classifier_values": None,
        "error_code": record["error_code"],
        "error_message": record["error_message"],
        "raw_response": record["response_body"],
    }
    if not record["success"]:
        return [base]

    payload = record["payload"] or {}
    pools = payload.get("customSQLPools") or []
    base["custom_sql_pools_enabled"] = payload.get("customSQLPoolsEnabled")
    base["pool_count"] = len(pools)
    if not pools:
        return [base]

    rows = []
    for pool in pools:
        classifier = pool.get("classifier") or {}
        row = dict(base)
        row.update(
            {
                "pool_name": pool.get("name"),
                "pool_is_default": pool.get("isDefault"),
                "pool_max_resource_percentage": pool.get("maxResourcePercentage"),
                "pool_optimize_for_reads": pool.get("optimizeForReads"),
                "classifier_type": classifier.get("type"),
                "classifier_values": classifier.get("value"),
            }
        )
        rows.append(row)
    return rows


rows = []
for index, (ws, _) in enumerate(target_workspaces, start=1):
    record = get_sql_pools_configuration(ws)
    rows.extend(inventory_rows(ws, record))
    flag = "OK " if record["success"] else "ERR"
    detail = (
        f"customSQLPoolsEnabled={(record['payload'] or {}).get('customSQLPoolsEnabled')}"
        if record["success"]
        else f"{record['status_code']} {record['error_code']}"
    )
    print(f"[{index}/{len(target_workspaces)}] {flag} {ws.get('displayName')} -> {detail}")

inventory_df = spark.createDataFrame(rows, schema=INVENTORY_SCHEMA)
target = write_delta(inventory_df, INVENTORY_TABLE)
print(f"\nWrote {inventory_df.count()} row(s) to {target}")
flush_api_log()

### Inventory — what did we find?

In [ ]:
inv = read_delta(INVENTORY_TABLE).where(f"run_id = '{RUN_ID}'")

print("Workspaces by 'custom SQL pools enabled' state:")
inv.select("workspace_id", "custom_sql_pools_enabled", "succeeded").distinct() \
   .groupBy("succeeded", "custom_sql_pools_enabled").count().show(truncate=False)

print("Pools found (i.e. workspaces with custom SQL pools enabled):")
display(
    inv.where("pool_name is not null")
       .select(
           "workspace_name",
           "custom_sql_pools_enabled",
           "pool_name",
           "pool_is_default",
           "pool_max_resource_percentage",
           "pool_optimize_for_reads",
           "classifier_type",
           "classifier_values",
       )
       .orderBy("workspace_name", "pool_name")
)

## 5. Enable custom SQL pools

For each in-scope workspace:
* **GET** the current configuration.
* If `customSQLPoolsEnabled` is already `true` → **do nothing** (unless `FORCE_UPDATE = True`).
* Otherwise **PATCH** a single default pool using the configured values
  (`optimizeForReads = False`, `maxResourcePercentage = 15` by default).

Every request and response is written to `sqlpools_api_log`; a per-workspace decision summary
goes to `sqlpools_enable_actions`.

> `DRY_RUN = True` in the config cell means nothing is changed — set it to `False` to apply.

In [ ]:
ACTION_SCHEMA = StructType(
    [
        StructField("run_id", StringType()),
        StructField("run_timestamp", TimestampType()),
        StructField("actioned_at", TimestampType()),
        StructField("workspace_id", StringType()),
        StructField("workspace_name", StringType()),
        StructField("dry_run", BooleanType()),
        StructField("force_update", BooleanType()),
        StructField("enabled_before", BooleanType()),
        StructField("pool_count_before", IntegerType()),
        StructField("action", StringType()),
        StructField("request_body", StringType()),
        StructField("status_code", IntegerType()),
        StructField("succeeded", BooleanType()),
        StructField("error_code", StringType()),
        StructField("error_message", StringType()),
        StructField("response_body", StringType()),
    ]
)


def build_update_payload():
    pool = {
        "name": POOL_NAME,
        "isDefault": POOL_IS_DEFAULT,
        "maxResourcePercentage": POOL_MAX_RESOURCE_PERCENTAGE,
        "optimizeForReads": POOL_OPTIMIZE_FOR_READS,
         "classifier": {
            "type": "Application Name",
            "value": [
            ]
         }
    }
    if POOL_CLASSIFIER:
        pool["classifier"] = POOL_CLASSIFIER
    return {"customSQLPoolsEnabled": CUSTOM_SQL_POOLS_ENABLED, "customSQLPools": [pool]}


PAYLOAD = build_update_payload()
print("Payload that will be PATCHed:")
print(json.dumps(PAYLOAD, indent=2))
print(f"\nDRY_RUN = {DRY_RUN}   FORCE_UPDATE = {FORCE_UPDATE}   workspaces in scope = {len(target_workspaces)}\n")

actions = []
for index, (ws, _) in enumerate(target_workspaces, start=1):
    label = f"[{index}/{len(target_workspaces)}] {ws.get('displayName')}"
    action = {
        "run_id": RUN_ID,
        "run_timestamp": RUN_TS,
        "actioned_at": utcnow(),
        "workspace_id": ws.get("id"),
        "workspace_name": ws.get("displayName"),
        "dry_run": DRY_RUN,
        "force_update": FORCE_UPDATE,
        "enabled_before": None,
        "pool_count_before": None,
        "action": None,
        "request_body": None,
        "status_code": None,
        "succeeded": None,
        "error_code": None,
        "error_message": None,
        "response_body": None,
    }

    current = get_sql_pools_configuration(ws)
    action["status_code"] = current["status_code"]
    if not current["success"]:
        action.update(
            {
                "action": "read-failed",
                "succeeded": False,
                "error_code": current["error_code"],
                "error_message": current["error_message"],
                "response_body": current["response_body"],
            }
        )
        print(f"{label}: READ FAILED ({current['status_code']} {current['error_code']})")
        actions.append(action)
        continue

    payload = current["payload"] or {}
    enabled_before = bool(payload.get("customSQLPoolsEnabled"))
    action["enabled_before"] = enabled_before
    action["pool_count_before"] = len(payload.get("customSQLPools") or [])

    if enabled_before and not FORCE_UPDATE:
        action.update({"action": "skipped-already-enabled", "succeeded": True})
        print(f"{label}: already enabled ({action['pool_count_before']} pool(s)) - no change")
        actions.append(action)
        continue

    action["request_body"] = json.dumps(PAYLOAD)
    if DRY_RUN:
        action.update({"action": "dry-run-would-enable", "succeeded": None})
        print(f"{label}: WOULD {'re-apply' if enabled_before else 'enable'} custom SQL pools (dry run)")
        actions.append(action)
        continue

    result = fabric_request(
        "PATCH",
        sql_pools_url(ws["id"]),
        body=PAYLOAD,
        operation="UpdateSqlPoolsConfiguration",
        workspace_id=ws.get("id"),
        workspace_name=ws.get("displayName"),
    )
    action.update(
        {
            "action": "re-applied" if enabled_before else "enabled",
            "status_code": result["status_code"],
            "succeeded": result["success"],
            "error_code": result["error_code"],
            "error_message": result["error_message"],
            "response_body": result["response_body"],
        }
    )
    if result["success"]:
        print(f"{label}: ENABLED (HTTP {result['status_code']})")
    else:
        action["action"] = "update-failed"
        print(f"{label}: UPDATE FAILED ({result['status_code']} {result['error_code']}) {result['error_message']}")
    actions.append(action)

actions_df = spark.createDataFrame(actions, schema=ACTION_SCHEMA)
target = write_delta(actions_df, ACTION_TABLE)
print(f"\nWrote {actions_df.count()} action row(s) to {target}")
flush_api_log()

display(
    actions_df.select(
        "workspace_name", "enabled_before", "action", "status_code", "succeeded", "error_message"
    ).orderBy("workspace_name")
)

## 6. Query the results later

The tables are ordinary Delta tables — query them from this notebook, the Lakehouse SQL
analytics endpoint, or a semantic model.

In [ ]:
print("Temp views:", register_views())

print("\n=== Latest state per workspace ===")
spark.sql(
    """
    SELECT workspace_name,
           custom_sql_pools_enabled,
           pool_name,
           pool_is_default,
           pool_max_resource_percentage,
           pool_optimize_for_reads,
           status_code,
           error_code
    FROM v_sqlpools_configuration
    WHERE run_timestamp = (SELECT MAX(run_timestamp) FROM v_sqlpools_configuration)
    ORDER BY workspace_name, pool_name
    """
).show(100, truncate=False)

print("=== Pools still optimised for reads ===")
spark.sql(
    """
    SELECT workspace_name, pool_name, pool_max_resource_percentage, run_timestamp
    FROM v_sqlpools_configuration
    WHERE pool_optimize_for_reads = true
    ORDER BY run_timestamp DESC, workspace_name
    """
).show(100, truncate=False)

print("=== Failed API calls (all runs) ===")
spark.sql(
    """
    SELECT logged_at, workspace_name, operation, method, status_code, error_code, error_message
    FROM v_sqlpools_api_log
    WHERE success = false
    ORDER BY logged_at DESC
    """
).show(100, truncate=False)

In [ ]:
df = spark.sql("SELECT * FROM v_sqlpools_api_log LIMIT 1000")
display(df)

In [ ]:
df = spark.sql("SELECT * FROM v_sqlpools_configuration LIMIT 1000")
display(df)

In [ ]:
df = spark.sql("SELECT * FROM v_sqlpools_enable_actions LIMIT 1000")
display(df)